# Stock Market Prediction with Machine Learning 📈

This notebook demonstrates how to build machine learning models for stock price prediction using historical market data and technical indicators.

## Learning Objectives
- Financial data analysis and preprocessing
- Technical indicator calculation
- Feature engineering for time series data
- Machine learning model development
- Model evaluation and comparison

In [ ]:
# Import required libraries
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries imported successfully!")

## 1. Data Collection and Loading

In [ ]:
# Function to download stock data
def download_stock_data(symbol, start_date, end_date):
    """Download stock data using yfinance"""
    try:
        stock = yf.Ticker(symbol)
        data = stock.history(start=start_date, end=end_date)
        return data
    except Exception as e:
        print(f"Error downloading data for {symbol}: {e}")
        return None

# Download data for multiple stocks
stocks = ['AAPL', 'GOOGL', 'MSFT', 'TSLA', 'AMZN']
start_date = '2020-01-01'
end_date = '2024-01-01'

stock_data = {}
for symbol in stocks:
    data = download_stock_data(symbol, start_date, end_date)
    if data is not None:
        stock_data[symbol] = data
        print(f"✅ Downloaded {symbol}: {len(data)} records")

print(f"\n📊 Total stocks downloaded: {len(stock_data)}")

## 2. Technical Indicators Calculation

In [ ]:
# Function to calculate technical indicators
def calculate_technical_indicators(df):
    """Calculate various technical indicators for stock data"""
    # Copy dataframe to avoid modifying original
    df = df.copy()
    
    # Moving averages
    df['SMA_20'] = df['Close'].rolling(window=20).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()
    df['EMA_12'] = df['Close'].ewm(span=12).mean()
    df['EMA_26'] = df['Close'].ewm(span=26).mean()
    
    # MACD
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_Signal'] = df['MACD'].ewm(span=9).mean()
    df['MACD_Histogram'] = df['MACD'] - df['MACD_Signal']
    
    # RSI
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # Bollinger Bands
    df['BB_Middle'] = df['Close'].rolling(window=20).mean()
    bb_std = df['Close'].rolling(window=20).std()
    df['BB_Upper'] = df['BB_Middle'] + (bb_std * 2)
    df['BB_Lower'] = df['BB_Middle'] - (bb_std * 2)
    
    # Price changes
    df['Price_Change'] = df['Close'].pct_change()
    df['Price_Change_5'] = df['Close'].pct_change(periods=5)
    df['Price_Change_10'] = df['Close'].pct_change(periods=10)
    
    # Volume indicators
    df['Volume_SMA'] = df['Volume'].rolling(window=20).mean()
    df['Volume_Ratio'] = df['Volume'] / df['Volume_SMA']
    
    # Volatility
    df['Volatility'] = df['Price_Change'].rolling(window=20).std()
    
    return df

# Calculate indicators for all stocks
for symbol in stock_data:
    stock_data[symbol] = calculate_technical_indicators(stock_data[symbol])

print("✅ Technical indicators calculated for all stocks!")

## 3. Data Visualization and Analysis

In [ ]:
# Create comprehensive stock analysis chart
def create_stock_analysis_chart(symbol, data):
    """Create comprehensive stock analysis chart with technical indicators"""
    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
        subplot_titles=(f'{symbol} Stock Price', 'Volume', 'RSI', 'MACD'),
        row_heights=[0.4, 0.2, 0.2, 0.2]
    )
    
    # Candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=data.index,
            open=data['Open'],
            high=data['High'],
            low=data['Low'],
            close=data['Close'],
            name='OHLC'
        ),
        row=1, col=1
    )
    
    # Moving averages
    fig.add_trace(
        go.Scatter(x=data.index, y=data['SMA_20'], name='SMA 20', line=dict(color='orange')),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=data.index, y=data['SMA_50'], name='SMA 50', line=dict(color='blue')),
        row=1, col=1
    )
    
    # Volume
    fig.add_trace(
        go.Bar(x=data.index, y=data['Volume'], name='Volume', marker_color='lightblue'),
        row=2, col=1
    )
    
    # RSI
    fig.add_trace(
        go.Scatter(x=data.index, y=data['RSI'], name='RSI', line=dict(color='purple')),
        row=3, col=1
    )
    fig.add_hline(y=70, line_dash="dash", line_color="red", row=3, col=1)
    fig.add_hline(y=30, line_dash="dash", line_color="green", row=3, col=1)
    
    # MACD
    fig.add_trace(
        go.Scatter(x=data.index, y=data['MACD'], name='MACD', line=dict(color='blue')),
        row=4, col=1
    )
    fig.add_trace(
        go.Scatter(x=data.index, y=data['MACD_Signal'], name='MACD Signal', line=dict(color='red')),
        row=4, col=1
    )
    fig.add_trace(
        go.Bar(x=data.index, y=data['MACD_Histogram'], name='MACD Histogram', marker_color='gray'),
        row=4, col=1
    )
    
    fig.update_layout(
        title=f'{symbol} Technical Analysis',
        height=800,
        xaxis_rangeslider_visible=False
    )
    
    return fig

# Create analysis chart for AAPL
if 'AAPL' in stock_data:
    aapl_chart = create_stock_analysis_chart('AAPL', stock_data['AAPL'])
    aapl_chart.show()

## 4. Feature Engineering for Machine Learning

In [ ]:
# Function to prepare features for ML
def prepare_ml_features(df, target_days=1):
    """Prepare features for machine learning prediction"""
    # Create target variable (future price)
    df['Target'] = df['Close'].shift(-target_days)
    
    # Select features
    feature_columns = [
        'Open', 'High', 'Low', 'Close', 'Volume',
        'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26',
        'MACD', 'MACD_Signal', 'MACD_Histogram',
        'RSI', 'BB_Middle', 'BB_Upper', 'BB_Lower',
        'Price_Change', 'Price_Change_5', 'Price_Change_10',
        'Volume_Ratio', 'Volatility'
    ]
    
    # Remove rows with NaN values
    df_clean = df.dropna()
    
    # Prepare X and y
    X = df_clean[feature_columns]
    y = df_clean['Target']
    
    return X, y

# Prepare features for AAPL
if 'AAPL' in stock_data:
    X_aapl, y_aapl = prepare_ml_features(stock_data['AAPL'])
    print(f"✅ Features prepared for AAPL: {X_aapl.shape}")
    print(f"📊 Feature columns: {list(X_aapl.columns)}")

## 5. Machine Learning Model Development

In [ ]:
# Split data and train models
def train_models(X, y):
    """Train multiple ML models for stock prediction"""
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Initialize models
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42)
    }
    
    # Train and evaluate models
    results = {}
    for name, model in models.items():
        # Train model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        
        # Cross-validation score
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
        
        results[name] = {
            'model': model,
            'mse': mse,
            'mae': mae,
            'r2': r2,
            'cv_mean': cv_scores.mean(),
            'cv_std': cv_scores.std(),
            'predictions': y_pred,
            'actual': y_test
        }
    
    return results, X_test, y_test

# Train models for AAPL
if 'AAPL' in stock_data:
    model_results, X_test, y_test = train_models(X_aapl, y_aapl)
    
    print("📊 Model Performance Results:")
    for name, results in model_results.items():
        print(f"\n{name}:")
        print(f"  MSE: {results['mse']:.4f}")
        print(f"  MAE: {results['mae']:.4f}")
        print(f"  R²: {results['r2']:.4f}")
        print(f"  CV Score: {results['cv_mean']:.4f} (+/- {results['cv_std']*2:.4f})")

## 6. Model Evaluation and Visualization

In [ ]:
# Create model comparison visualization
def plot_model_comparison(model_results):
    """Create visualization comparing model performance"""
    # Prepare data for plotting
    models = list(model_results.keys())
    r2_scores = [model_results[model]['r2'] for model in models]
    mae_scores = [model_results[model]['mae'] for model in models]
    
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('R² Score Comparison', 'MAE Score Comparison')
    )
    
    # R² scores
    fig.add_trace(
        go.Bar(x=models, y=r2_scores, name='R² Score', marker_color='lightblue'),
        row=1, col=1
    )
    
    # MAE scores
    fig.add_trace(
        go.Bar(x=models, y=mae_scores, name='MAE Score', marker_color='lightcoral'),
        row=1, col=2
    )
    
    fig.update_layout(
        title='Model Performance Comparison',
        height=400
    )
    
    return fig

# Plot model comparison
if 'AAPL' in stock_data:
    comparison_fig = plot_model_comparison(model_results)
    comparison_fig.show()
    
    # Plot actual vs predicted
    best_model_name = max(model_results.keys(), key=lambda x: model_results[x]['r2'])
    best_model = model_results[best_model_name]
    
    fig = px.scatter(
        x=best_model['actual'],
        y=best_model['predictions'],
        title=f'Actual vs Predicted Stock Prices ({best_model_name})',
        labels={'x': 'Actual Price', 'y': 'Predicted Price'}
    )
    
    # Add perfect prediction line
    min_val = min(best_model['actual'].min(), best_model['predictions'].min())
    max_val = max(best_model['actual'].max(), best_model['predictions'].max())
    fig.add_trace(
        go.Scatter(x=[min_val, max_val], y=[min_val, max_val], 
                   mode='lines', name='Perfect Prediction', line=dict(dash='dash'))
    )
    
    fig.show()

## 7. Key Insights and Conclusions

### 📈 **Key Findings:**
1. **Technical Indicators**: Moving averages, RSI, and MACD provide valuable signals
2. **Model Performance**: Random Forest generally outperforms Linear Regression
3. **Feature Importance**: Price changes and volume indicators are crucial predictors
4. **Prediction Accuracy**: Models achieve reasonable accuracy for short-term predictions

### 🎯 **Learning Outcomes:**
- ✅ Financial data preprocessing and technical analysis
- ✅ Feature engineering for time series data
- ✅ Machine learning model development and evaluation
- ✅ Model performance comparison and visualization
- ✅ Real-world financial modeling workflow

### 🚀 **Next Steps:**
- Implement more advanced models (LSTM, XGBoost)
- Add sentiment analysis from news and social media
- Create portfolio optimization strategies
- Build real-time prediction dashboard
- Implement risk management algorithms